# Lesson 27 — Machine Learning for Optimization

## Learning objectives

1. Distinguish *learning to branch*, *learning to cut*, *learning primal heuristics*.
2. Recognize the role of GNNs for instance representation {cite:p}`Gasse2019`.
3. Understand decision-focused vs predict-then-optimize learning {cite:p}`Elmachtoub2022,Wilder2019`.
4. Critically read a "ML beats Gurobi" paper.

## 1. The taxonomy

{cite:p}`Bengio2021` organize ML-for-optimization into:

- **End-to-end learning**: predict the solution from instance features. Limited to easy problems.
- **Configuring** combinatorial algorithms: choose hyperparameters per instance.
- **Interleaved with B&B**: learn branching, cuts, heuristics inside the tree.

## 2. Learning to branch

Replace strong branching (Lesson 14) with a fast classifier trained to imitate it. {cite:p}`Gasse2019` use a GNN over the variable-constraint bipartite graph.

**Training:** supervised — at each B&B node, log strong-branching scores, train classifier to predict argmax. **Inference:** use the classifier instead of strong branching, getting strong-branching quality at fraction of the cost.

Reported speedups: 2–5x on benchmark MILPs.

## 3. Learning primal heuristics

Train a model to *propose* good integer-feasible incumbents from the LP relaxation. Often used early in B&B. {cite:p}`Bengio2021` survey several approaches.

## 4. Decision-focused learning

When the *output* of a predictive model is fed into an optimizer, training the predictor to minimize prediction error is suboptimal — what matters is the *decision quality*. {cite:p}`Wilder2019,Mandi2020,Elmachtoub2022` develop *decision-focused* training: differentiate the optimizer (Lesson 26) and propagate gradients.

## 5. Reading the literature critically

ML-for-optimization papers vary widely in claim quality. Checklist:

- **Compared to what?** "Beats Gurobi default" without solver tuning is weak.
- **Generalization?** Trained and tested on the same instance distribution? Or transfers?
- **Compute cost?** Often the ML method's training cost is omitted.
- **Statistical reporting?** Confidence intervals on benchmarks are rare in ML papers.

For a sober view, see {cite:p}`Lu2024,Sandrin2025`. The field is rapidly evolving; today's "breakthrough" is often tomorrow's "moderate improvement".

## References
{cite:p}`Bengio2021,Gasse2019,Elmachtoub2022,Wilder2019,Mandi2020`.

In [ ]:
# discopt course compat shim — installs `add_variable / add_variables /
# add_constraint / add_constraints / is_convex` and a friendly `.solve(...)`
# on `discopt.Model`. Run this cell first.
import sys, pathlib
_repo = pathlib.Path.cwd()
while _repo != _repo.parent and not (_repo / "course").is_dir(): _repo = _repo.parent
if str(_repo) not in sys.path: sys.path.insert(0, str(_repo))
from course._compat import *  # noqa: F401,F403


In [ ]:
import numpy as np, discopt as do

# Toy: train a model to predict "down child has more nodes"
# (would be plugged into a branching callback in a real system)
# Skipping training; we demonstrate the callback hook only.
m = do.Model("k20")
n = 20
rng = np.random.default_rng(0)
v = rng.integers(1, 30, n); w = rng.integers(1, 20, n); W = int(0.4 * w.sum())
x = m.add_variables(n, vtype="binary")
m.add_constraint(w @ x <= W); m.maximize(v @ x)

def my_branching(node):
    # Toy "ML" model: pick variable with largest LP fractional part times a feature
    feats = node.lp_fractional()
    return int(np.argmax(feats * (v / w)))

m.set_branching_callback(my_branching)
r = m.solve()
print("nodes:", r.nodes, " objective:", r.objective)
